In [8]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import re
from tabulate import tabulate

In [9]:
params_data = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine_data = create_engine(f"mssql+pyodbc:///?odbc_connect={params_data}")


query_data = """
SELECT DISTINCT 
    c.Cliente,
    v.Fecha AS Fecha_Venta,
    v.Canal AS Canal_Recompra,
	v.Categoria,
	v.Venta_Neta AS Venta
FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2025-05-01'
    AND v.Venta_Neta > 0
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_data, engine_data)

In [10]:
df_ventas

,Cliente,Fecha_Venta,Canal_Recompra,Categoria,Venta
0,C,2025-05-07,Shopify,ARTICULOS DE ASEO,82615.20
1,C,2025-05-07,Shopify,ARTICULOS DE ASEO,211755.60
2,C,2025-05-07,Shopify,CUIDADO CORPORAL,114855.30
3,C,2025-05-07,Shopify,CUIDADO FACIAL,138649.50
4,C,2025-05-08,Shopify,ARTICULOS DE ASEO,131927.10
...,...,...,...,...,...
134660,CZ7886894,2025-07-30,Retail,ARTICULOS DE ASEO,103361.34
134661,CZ7886894,2025-07-30,Retail,ARTICULOS DE ASEO,138655.46
134662,CZ7886894,2025-07-30,Retail,CUIDADO CORPORAL,203361.34
134663,CZ7886894,2025-07-30,Retail,CUIDADO DE MANOS,45378.15


In [11]:
# Normalizar el nombre del canal
df_ventas['Canal_Recompra'] = df_ventas['Canal_Recompra'].replace({
    'Chat Center': 'Social Selling',
    'CHATCENTER WEB': 'Social Selling'
})

df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

df_ventas

,Cliente,Fecha_Venta,Canal_Recompra,Categoria,Venta
0,C,2025-05-07,Shopify,ARTICULOS DE ASEO,82615.20
1,C,2025-05-07,Shopify,ARTICULOS DE ASEO,211755.60
2,C,2025-05-07,Shopify,CUIDADO CORPORAL,114855.30
3,C,2025-05-07,Shopify,CUIDADO FACIAL,138649.50
4,C,2025-05-08,Shopify,ARTICULOS DE ASEO,131927.10
...,...,...,...,...,...
134660,CZ7886894,2025-07-30,Retail,ARTICULOS DE ASEO,103361.34
134661,CZ7886894,2025-07-30,Retail,ARTICULOS DE ASEO,138655.46
134662,CZ7886894,2025-07-30,Retail,CUIDADO CORPORAL,203361.34
134663,CZ7886894,2025-07-30,Retail,CUIDADO DE MANOS,45378.15


In [12]:
# Cargar el archivo Excel
ruta_excel = 'PLAN 120.xlsx'
plan120_excel = pd.read_excel(ruta_excel,sheet_name=None)

In [13]:
# ▶️ Inicializar diccionario de resultados
resultados = {}

# ▶️ Recorrer cada hoja (mes) del Plan 120
for mes, df_nuevos in plan120_excel.items():

    df_nuevos['Fecha_Ingreso'] = pd.to_datetime(df_nuevos['Fecha_Ingreso']).dt.date

    # ▶️ Unir con df_ventas para quedarnos solo con ventas posteriores al ingreso
    df_merge = df_ventas.merge(df_nuevos, on='Cliente', how='inner')

    # Filtrar ventas después de la fecha de ingreso
    df_recompras = df_merge[df_merge['Fecha_Venta'] > df_merge['Fecha_Ingreso']].copy()

    # Paso 3: Obtener primera fecha de recompra por cliente
    primeras_fechas = df_recompras.groupby('Cliente')['Fecha_Venta'].min().reset_index()
    primeras_fechas.rename(columns={'Fecha_Venta': 'FechaRecompra'}, inplace=True)

    # Paso 4: Unir con df_recompras para quedarnos con solo compras del mismo día
    df_primera_recompra = df_recompras.merge(primeras_fechas, on='Cliente')
    df_primera_recompra = df_primera_recompra[
        df_primera_recompra['Fecha_Venta'] == df_primera_recompra['FechaRecompra']
    ].copy()

    # Agregar fecha de ingreso
    df_primera_recompra = df_nuevos[['Cliente']].merge(
        df_primera_recompra, on='Cliente', how='inner'
    )

    # Agrupar por cliente + canal + categoría
    df_final = df_primera_recompra.groupby(
        ['Cliente', 'Fecha_Ingreso', 'Canal_Ingreso','FechaRecompra', 'Canal_Recompra', 'Categoria'], as_index=False
    )['Venta'].sum()

    # Marcar cambio de canal
    df_final['CambioCanal'] = df_final.apply(
        lambda row: 'NO CAMBIO' if row['Canal_Ingreso'] == row['Canal_Recompra'] else 'CAMBIO',
        axis=1
    )

     # Guardar resultado final
    resultados[mes] = df_final

     # 📊 Reporte de resumen por mes
    total_recompradores = df_final['Cliente'].nunique()
    cambios = df_final[df_final['CambioCanal'] == 'CAMBIO']['Cliente'].nunique()
    sin_cambio = total_recompradores - cambios

    print(f"\n📅 MES: {mes.upper()}")
    print(f"➡️ Clientes que recompraron: {total_recompradores}")
    print(f"🔁 Clientes que CAMBIARON de canal: {cambios}")
    print(f"✅ Clientes que NO cambiaron de canal: {sin_cambio}")

    # Detalle por cambio de canal
    detalle_cambios = df_final[df_final['CambioCanal'] == 'CAMBIO'].groupby(
        ['Canal_Ingreso', 'Canal_Recompra']
    )['Cliente'].nunique().reset_index(name='Clientes')

    if not detalle_cambios.empty:
        print("\n🔎 Detalle de cambios de canal:")
        print(tabulate(detalle_cambios, headers='keys', tablefmt='tsv', showindex=False))
    else:
        print("\n✔️ No hubo cambios de canal.")

    # ▶️ Agregar columna con nombre del mes de la recompra
    df_final['MesRecompra'] = pd.to_datetime(df_final['FechaRecompra']).dt.strftime('%B').str.upper()

    # ▶️ Calcular resumen de clientes únicos por mes de recompra
    recuento_clientes = df_final.groupby('MesRecompra')['Cliente'].nunique().reset_index()
    recuento_clientes.columns = ['MesRecompra', 'ClientesUnicos']

    print(f"\n📆 Recompras para Plan 120 ({mes.upper()}):")
    print(tabulate(recuento_clientes, headers='keys', tablefmt='tsv', showindex=False))


📅 MES: JUNIO 
➡️ Clientes que recompraron: 271
🔁 Clientes que CAMBIARON de canal: 37
✅ Clientes que NO cambiaron de canal: 234

🔎 Detalle de cambios de canal:
Canal_Ingreso  	Canal_Recompra  	  Clientes
Retail         	Ferias          	         2
Retail         	Shopify         	        13
Retail         	Social Selling  	         2
Retail         	Whatsapp        	         1
Shopify        	Retail          	        13
Shopify        	Social Selling  	         2
Shopify        	Whatsapp        	         1
Social Selling 	Retail          	         1
Social Selling 	Shopify         	         2

📆 Recompras para Plan 120 (JUNIO ):
MesRecompra  	  ClientesUnicos
AUGUST       	              53
JULY         	              74
JUNE         	              58
NOVEMBER     	               3
OCTOBER      	              36
SEPTEMBER    	              47

📅 MES: JULIO
➡️ Clientes que recompraron: 241
🔁 Clientes que CAMBIARON de canal: 25
✅ Clientes que NO cambiaron de canal: 216

🔎 Detalle de cambi

In [14]:
# ▶️ Exportar a Excel final
with pd.ExcelWriter("plan_120_recompras_detallado.xlsx") as writer:
    for mes, df in resultados.items():
        df.to_excel(writer, sheet_name=mes[:31], index=False)
